
# 06 — Tanore & Manda Leave-One-Date-Out Ablation

এই একটি notebook Tanore এবং Manda—দুই area-তেই **All 4 dates, Without Jan, Without Mar, Without Apr1, Without Apr2** run করবে।

প্রতিটি area-তে:
- FusedHybrid + RandomForest
- FusedHybrid + XGBoost
- PlanetOnly + RandomForest
- PlanetOnly + XGBoost

**Publication-safe rule:** hyperparameters শুধু full-date training data দিয়ে tune হবে এবং ablation run-এ freeze থাকবে। Removed date-এর information derived features-এ leak না করার জন্য NDVI summary/change features প্রতিবার remaining dates থেকে নতুন করে তৈরি হবে।


### GitHub execution note
This notebook preserves the publication analysis logic. Local absolute paths were replaced with the portable `BORO_PROJECT_ROOT` setting. Run Jupyter from the repository root or set that environment variable before execution. Generated figures and tables are written below `Outputs/`; licensed source imagery is not included.


In [ ]:

# CELL 1 — Imports, paths, settings
from pathlib import Path
import json, os, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, cohen_kappa_score, confusion_matrix, f1_score,
    matthews_corrcoef, precision_recall_curve, precision_score,
    recall_score, roc_auc_score
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold, cross_val_predict
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path(os.environ.get('BORO_PROJECT_ROOT', str(Path.cwd()))).expanduser().resolve()
OUTPUT_ROOT = PROJECT_ROOT / 'Outputs' / 'Q1_Extensions' / 'Leave_One_Date_Out_Ablation'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
for f in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    f.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
CV_SPLITS = 5
RF_SEARCH_ITERATIONS = 15
XGB_SEARCH_ITERATIONS = 20
BOOTSTRAP_REPLICATES = 1000
QUICK_MODE = False
if QUICK_MODE:
    RF_SEARCH_ITERATIONS = 3
    XGB_SEARCH_ITERATIONS = 3
    BOOTSTRAP_REPLICATES = 200

AREAS=['Tanore','Manda']
STREAMS=['FusedHybrid','PlanetOnly']
MODELS=['RandomForest','XGBoost']
DATE_ORDER=['Jan','Mar','Apr1','Apr2']
BAND_NAMES=['Blue','Green','Red','NIR']
ABLATION_CONDITIONS={
    'All_4_Dates':['Jan','Mar','Apr1','Apr2'],
    'Without_Jan':['Mar','Apr1','Apr2'],
    'Without_Mar':['Jan','Apr1','Apr2'],
    'Without_Apr1':['Jan','Mar','Apr2'],
    'Without_Apr2':['Jan','Mar','Apr1'],
}
print('Project root :', PROJECT_ROOT)
print('Output folder:', OUTPUT_ROOT)
print('Quick mode   :', QUICK_MODE)


In [ ]:

# CELL 2 — Load final training/validation sample tables
DIRECT_FEATURES=[f'{d}_{b}' for d in DATE_ORDER for b in BAND_NAMES] + [f'{d}_NDVI' for d in DATE_ORDER]

def table_path(area,stream,split):
    return PROJECT_ROOT/'Outputs'/area/'Classification_Q1'/'tables'/f'Q1_{stream}_{split}_Samples.csv'

def load_table(area,stream,split):
    p=table_path(area,stream,split)
    if not p.exists():
        raise FileNotFoundError(f'Required file not found:\n{p}\nRun final 03_{area}_Q1_Classification first.')
    df=pd.read_csv(p)
    required={'sample_id','class','group',*DIRECT_FEATURES}
    missing=sorted(required-set(df.columns))
    if missing:
        raise ValueError(f'{p.name} missing columns: {missing}')
    if df[DIRECT_FEATURES].isna().any().any():
        raise ValueError(f'{p.name} contains missing direct predictor values.')
    return df.copy()

sample_tables={a:{s:{sp:load_table(a,s,sp) for sp in ['Training','Validation']} for s in STREAMS} for a in AREAS}
rows=[]
for a in AREAS:
    for s in STREAMS:
        for sp in ['Training','Validation']:
            d=sample_tables[a][s][sp]
            rows.append({'area':a,'stream':s,'split':sp,'n':len(d),'rice':int((d['class']==1).sum()),'nonrice':int((d['class']==0).sum()),'groups':d['group'].astype(str).nunique()})
input_summary=pd.DataFrame(rows)
display(input_summary)
print('✅ Input tables found.')


In [ ]:

# CELL 3 — Leakage-safe feature rebuilding for each date condition
DATE_POSITION={'Jan':0,'Mar':1,'Apr1':2,'Apr2':3}

def build_ablation_features(df,active_dates):
    active_dates=list(active_dates)
    ff=pd.DataFrame(index=df.index)
    names=[]
    # direct bands + NDVI
    for d in active_dates:
        cols=[f'{d}_{b}' for b in BAND_NAMES]+[f'{d}_NDVI']
        for c in cols:
            ff[c]=df[c].astype('float32')
        names.extend(cols)
    # summaries rebuilt from remaining dates only
    ndvi_cols=[f'{d}_NDVI' for d in active_dates]
    arr=df[ndvi_cols].to_numpy(dtype='float32')
    ff['NDVI_mean']=arr.mean(axis=1)
    ff['NDVI_std']=arr.std(axis=1)
    ff['NDVI_min']=arr.min(axis=1)
    ff['NDVI_max']=arr.max(axis=1)
    ff['NDVI_amplitude']=ff['NDVI_max']-ff['NDVI_min']
    peak_idx=np.argmax(arr,axis=1)
    pos=np.array([DATE_POSITION[d] for d in active_dates],dtype='float32')
    ff['NDVI_peak_timing']=pos[peak_idx]
    names += ['NDVI_mean','NDVI_std','NDVI_min','NDVI_max','NDVI_amplitude','NDVI_peak_timing']
    # consecutive changes using remaining dates only
    for early,later in zip(active_dates[:-1],active_dates[1:]):
        n=f'dNDVI_{later}_{early}'
        ff[n]=(df[f'{later}_NDVI']-df[f'{early}_NDVI']).astype('float32')
        names.append(n)
    # whole available-season change
    if len(active_dates)>=3:
        n='dNDVI_season'
        ff[n]=(df[f'{active_dates[-1]}_NDVI']-df[f'{active_dates[0]}_NDVI']).astype('float32')
        names.append(n)
    X=ff[names].to_numpy(dtype='float32')
    if not np.isfinite(X).all():
        raise ValueError('Non-finite values found in ablation features.')
    return X,names

schema=[]
for cond,dates in ABLATION_CONDITIONS.items():
    _,names=build_ablation_features(sample_tables['Tanore']['PlanetOnly']['Training'],dates)
    schema.append({'condition':cond,'active_dates':', '.join(dates),'feature_count':len(names),'features':', '.join(names)})
feature_schema=pd.DataFrame(schema)
display(feature_schema[['condition','active_dates','feature_count']])
print('✅ Removed-date information is excluded from rebuilt features.')


In [ ]:

# CELL 4 — CV, tuning, threshold, metrics

def make_cv(y,groups):
    gt=pd.DataFrame({'group':groups,'class':y}).drop_duplicates()
    counts=gt.groupby('class')['group'].nunique()
    n=min(CV_SPLITS,int(counts.min()),int(gt['group'].nunique()))
    if n<3: raise ValueError('Need at least 3 independent groups per class.')
    return StratifiedGroupKFold(n_splits=n,shuffle=True,random_state=RANDOM_SEED)

def best_f1_threshold(y,p):
    pr,re,th=precision_recall_curve(y,p)
    if len(th)==0: return 0.5
    f=2*pr[:-1]*re[:-1]/np.maximum(pr[:-1]+re[:-1],1e-12)
    return float(th[int(np.nanargmax(f))])

def estimator_space(model_name,y):
    if model_name=='RandomForest':
        est=RandomForestClassifier(random_state=RANDOM_SEED,class_weight='balanced',n_jobs=1)
        space={'n_estimators':[300,500,800,1000],'max_depth':[None,10,15,20,30],'min_samples_split':[2,5,10],'min_samples_leaf':[1,2,4,8],'max_features':['sqrt','log2',0.4,0.7],'bootstrap':[True,False]}
        niter=RF_SEARCH_ITERATIONS
    else:
        neg=max(int((y==0).sum()),1); pos=max(int((y==1).sum()),1)
        est=XGBClassifier(objective='binary:logistic',eval_metric='logloss',tree_method='hist',random_state=RANDOM_SEED,n_jobs=1,scale_pos_weight=neg/pos)
        space={'n_estimators':[300,500,700,1000],'max_depth':[3,4,5,6,8],'learning_rate':[0.01,0.03,0.05,0.08,0.1],'subsample':[0.65,0.8,0.9,1.0],'colsample_bytree':[0.6,0.75,0.9,1.0],'min_child_weight':[1,3,5,8],'gamma':[0.0,0.1,0.3],'reg_alpha':[0.0,0.01,0.1,0.5],'reg_lambda':[0.5,1.0,2.0,5.0]}
        niter=XGB_SEARCH_ITERATIONS
    if QUICK_MODE:
        niter=3
    return est,space,niter

def tune_full(model_name,X,y,groups):
    est,space,niter=estimator_space(model_name,y)
    cv=make_cv(y,groups)
    search=RandomizedSearchCV(est,space,n_iter=niter,scoring='average_precision',n_jobs=-1,cv=cv,random_state=RANDOM_SEED,refit=True,verbose=1)
    search.fit(X,y,groups=groups)
    return {'best_params':search.best_params_,'best_cv_ap':float(search.best_score_)}

def frozen_estimator(model_name,y,params):
    est,_,_=estimator_space(model_name,y)
    est.set_params(**params)
    return est

def train_eval_condition(model_name,Xtr,ytr,gtr,Xv,yv,params):
    cv=make_cv(ytr,gtr)
    est=frozen_estimator(model_name,ytr,params)
    oof=cross_val_predict(est,Xtr,ytr,groups=gtr,cv=cv,method='predict_proba',n_jobs=-1)[:,1]
    thr=best_f1_threshold(ytr,oof)
    final=clone(est).fit(Xtr,ytr)
    prob=final.predict_proba(Xv)[:,1]
    pred=(prob>=thr).astype('uint8')
    return thr,prob,pred

def metrics(y,p,pr):
    tn,fp,fn,tp=confusion_matrix(y,pr,labels=[0,1]).ravel()
    return {
        'n':len(y),'OA':accuracy_score(y,pr),'balanced_accuracy':balanced_accuracy_score(y,pr),
        'precision':precision_score(y,pr,zero_division=0),'recall':recall_score(y,pr,zero_division=0),
        'specificity':tn/(tn+fp) if (tn+fp) else np.nan,'F1':f1_score(y,pr,zero_division=0),
        'kappa':cohen_kappa_score(y,pr),'MCC':matthews_corrcoef(y,pr),
        'ROC_AUC':roc_auc_score(y,p),'PR_AUC':average_precision_score(y,p),'Brier':brier_score_loss(y,p),
        'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp)
    }

def bootstrap_f1(y,pred,groups,reps=1000):
    d=pd.DataFrame({'y':y,'pred':pred,'g':groups}).reset_index(drop=True)
    ug=d['g'].astype(str).drop_duplicates().to_numpy(); rng=np.random.default_rng(RANDOM_SEED); vals=[]
    for _ in range(reps):
        gs=rng.choice(ug,size=len(ug),replace=True)
        idx=np.concatenate([d.index[d['g'].astype(str)==str(g)].to_numpy() for g in gs])
        vals.append(f1_score(d.loc[idx,'y'],d.loc[idx,'pred'],zero_division=0))
    return tuple(np.quantile(vals,[0.025,0.975]))



## 75 fits vs 100 fits এখানে কীভাবে ব্যবহার হচ্ছে
RF: **15 candidates × 5 folds = 75 fits**  
XGBoost: **20 candidates × 5 folds = 100 fits**

এগুলো শুধু full-date model tuning-এর জন্য। Ablation condition-এ hyperparameters আর নতুন করে search করা হবে না।


In [ ]:

# CELL 5 — Tune once on full four-date training data
frozen={}; tuning_rows=[]
for area in AREAS:
    frozen[area]={}
    for stream in STREAMS:
        frozen[area][stream]={}
        tr=sample_tables[area][stream]['Training']
        Xfull,full_names=build_ablation_features(tr,ABLATION_CONDITIONS['All_4_Dates'])
        y=tr['class'].to_numpy(dtype=int); g=tr['group'].astype(str).to_numpy()
        for model in MODELS:
            print('\n'+'='*76); print('FULL-DATE TUNING:',area,'|',stream,'|',model); print('='*76)
            r=tune_full(model,Xfull,y,g)
            frozen[area][stream][model]=r
            tuning_rows.append({'area':area,'stream':stream,'model':model,'full_date_feature_count':len(full_names),'best_cv_average_precision':r['best_cv_ap'],'best_parameters':json.dumps(r['best_params'])})
            print('Best CV AP:',round(r['best_cv_ap'],4))
tuning_table=pd.DataFrame(tuning_rows)
display(tuning_table)
print('✅ Full-date tuning complete; hyperparameters are now frozen.')


In [ ]:

# CELL 6 — Run all ablation conditions
result_rows=[]; pred_frames=[]
for area in AREAS:
    for stream in STREAMS:
        tr=sample_tables[area][stream]['Training']; va=sample_tables[area][stream]['Validation']
        ytr=tr['class'].to_numpy(dtype=int); gtr=tr['group'].astype(str).to_numpy()
        yv=va['class'].to_numpy(dtype=int); gv=va['group'].astype(str).to_numpy()
        for cond,dates in ABLATION_CONDITIONS.items():
            Xtr,names=build_ablation_features(tr,dates); Xv,names2=build_ablation_features(va,dates)
            if names!=names2: raise RuntimeError('Feature schema mismatch.')
            for model in MODELS:
                print('\n'+'='*76); print(area,'|',stream,'|',model,'|',cond); print('Active dates:',dates,'| Features:',len(names)); print('='*76)
                params=frozen[area][stream][model]['best_params']
                thr,prob,pred=train_eval_condition(model,Xtr,ytr,gtr,Xv,yv,params)
                m=metrics(yv,prob,pred); lo,hi=bootstrap_f1(yv,pred,gv,BOOTSTRAP_REPLICATES)
                removed='' if cond=='All_4_Dates' else cond.replace('Without_','')
                result_rows.append({'area':area,'stream':stream,'model':model,'condition':cond,'date_removed':removed,'active_dates':', '.join(dates),'feature_count':len(names),'threshold_training_only':thr,'F1_CI_low':lo,'F1_CI_high':hi,**m})
                pred_frames.append(pd.DataFrame({'sample_id':va['sample_id'].astype(str),'group':va['group'].astype(str),'true_class':yv,'probability':prob,'prediction':pred,'area':area,'stream':stream,'model':model,'condition':cond}))
                print('F1:',round(m['F1'],4),'| MCC:',round(m['MCC'],4),'| OA:',round(m['OA'],4))
results=pd.DataFrame(result_rows); predictions=pd.concat(pred_frames,ignore_index=True)
print('\n✅ ALL ABLATION RUNS FINISHED')


In [ ]:

# CELL 7 — Compute drop relative to full-date baseline
base=results[results['condition']=='All_4_Dates'][['area','stream','model','F1','MCC','OA','ROC_AUC','PR_AUC','Brier']].rename(columns={'F1':'baseline_F1','MCC':'baseline_MCC','OA':'baseline_OA','ROC_AUC':'baseline_ROC_AUC','PR_AUC':'baseline_PR_AUC','Brier':'baseline_Brier'})
ablation=results.merge(base,on=['area','stream','model'],how='left')
ablation['F1_drop_vs_full']=ablation['baseline_F1']-ablation['F1']
ablation['MCC_drop_vs_full']=ablation['baseline_MCC']-ablation['MCC']
ablation['OA_drop_vs_full']=ablation['baseline_OA']-ablation['OA']
display(ablation[['area','stream','model','condition','F1','baseline_F1','F1_drop_vs_full','MCC','MCC_drop_vs_full','OA','OA_drop_vs_full']].round(4))


In [ ]:

# CELL 8 — Overall phenological date ranking
omit=ablation[ablation['date_removed']!=''].copy()
rank=(omit.groupby('date_removed',as_index=False).agg(
    mean_F1_drop=('F1_drop_vs_full','mean'),median_F1_drop=('F1_drop_vs_full','median'),
    mean_MCC_drop=('MCC_drop_vs_full','mean'),mean_OA_drop=('OA_drop_vs_full','mean'),
    min_F1_drop=('F1_drop_vs_full','min'),max_F1_drop=('F1_drop_vs_full','max'))
    .sort_values('mean_F1_drop',ascending=False).reset_index(drop=True))
rank.insert(0,'importance_rank',np.arange(1,len(rank)+1))
print('Higher positive mean_F1_drop = more important date.')
display(rank.round(4))


In [ ]:

# CELL 9 — Figures
order=['All_4_Dates','Without_Jan','Without_Mar','Without_Apr1','Without_Apr2']
for area in AREAS:
    for stream in STREAMS:
        for model in MODELS:
            s=ablation[(ablation.area==area)&(ablation.stream==stream)&(ablation.model==model)].set_index('condition').reindex(order).reset_index()
            fig,ax=plt.subplots(figsize=(8.5,5))
            ax.bar(s['condition'],s['F1'])
            ax.set_ylabel('Independent validation F1')
            ax.set_title(f'{area} | {stream} | {model}\nLeave-One-Date-Out Ablation')
            ax.tick_params(axis='x',rotation=25)
            ax.set_ylim(max(0.0,float(s['F1'].min())-0.08),1.01)
            fig.tight_layout(); fig.savefig(FIGURE_DIR/f'Ablation_F1_{area}_{stream}_{model}.png',dpi=300,bbox_inches='tight'); plt.close(fig)
fig,ax=plt.subplots(figsize=(7.5,4.8))
ax.bar(rank['date_removed'],rank['mean_F1_drop'])
ax.axhline(0,linewidth=1)
ax.set_xlabel('Removed observation date'); ax.set_ylabel('Mean F1 drop when date is removed'); ax.set_title('Overall Phenological Date Importance')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'Overall_Date_Importance_Mean_F1_Drop.png',dpi=300,bbox_inches='tight'); plt.show()
print('✅ Figures saved.')


In [ ]:

# CELL 10 — Save outputs
input_summary.to_csv(TABLE_DIR/'Ablation_Input_Summary.csv',index=False)
feature_schema.to_csv(TABLE_DIR/'Ablation_Feature_Schema.csv',index=False)
tuning_table.to_csv(TABLE_DIR/'Ablation_FullDate_Frozen_Hyperparameters.csv',index=False)
ablation.to_csv(TABLE_DIR/'Leave_One_Date_Out_Ablation_Metrics.csv',index=False)
rank.to_csv(TABLE_DIR/'Date_Importance_Ranking.csv',index=False)
predictions.to_csv(TABLE_DIR/'Ablation_Validation_Predictions.csv',index=False)
excel=OUTPUT_ROOT/'Leave_One_Date_Out_Ablation_Results.xlsx'
with pd.ExcelWriter(excel,engine='openpyxl') as w:
    input_summary.to_excel(w,sheet_name='Input_Summary',index=False)
    feature_schema.to_excel(w,sheet_name='Feature_Schema',index=False)
    tuning_table.to_excel(w,sheet_name='Frozen_Hyperparameters',index=False)
    ablation.to_excel(w,sheet_name='Ablation_Metrics',index=False)
    rank.to_excel(w,sheet_name='Date_Importance',index=False)
    predictions.to_excel(w,sheet_name='Predictions',index=False)
print('\n'+'='*78)
print('✅ LEAVE-ONE-DATE-OUT ABLATION COMPLETE')
print('='*78)
print('Excel  :',excel)
print('Tables :',TABLE_DIR)
print('Figures:',FIGURE_DIR)



## Interpretation

`F1_drop_vs_full` সবচেয়ে গুরুত্বপূর্ণ:
- **Positive** → date বাদ দিলে performance কমেছে → date important.
- **Large positive** → date খুব important.
- **Near 0** → date বাদ দিলেও প্রায় একই.
- **Negative** → date বাদ দিলে performance বেড়েছে → date redundant/noisy হতে পারে.

### Publication wording
> Leave-one-date-out ablation was performed by retraining each classifier after removing one observation date at a time. Hyperparameters tuned on the complete four-date training set were held fixed across ablation experiments, while decision thresholds were recalibrated exclusively from training out-of-fold predictions. Temporal summary and change features were reconstructed using only the remaining dates to prevent information leakage from the omitted observation.
